# Notebook 08: Significance testing

Consolidates all inferential statistical tests into a unified report.

Methodology Summary:
- Direction Contrasts (SQ1 & SQ2): Bootstrap difference CIs across evaluation sets (AUROC, Standard ECE, Adaptive ECE).
- Architecture Contrasts: DeLong's test (AUROC) & McNemar's test (Accuracy) on paired evaluation sets.
- Size Interaction (SQ3): Bootstrapped direction-gap comparison across source sizes (Size 190 vs Size 50).
- Multiple Testing Correction: Holm-Bonferroni step-down adjustment applied to p-value tests.


## 1. Setup and Load Predictions

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import sys
from pathlib import Path

REPO_ON_DRIVE = "/content/drive/MyDrive/crosspop-cxr-asymmetry"
for candidate in [REPO_ON_DRIVE, "..", "."]:
    p = Path(candidate).resolve()
    if (p / "config.py").exists():
        sys.path.insert(0, str(p))
        break

import importlib
import numpy as np
import pandas as pd

import config
from src import metrics as MET
from src import stats as ST
from src import results as RES

for mod in (config, MET, ST, RES):
    importlib.reload(mod)

config.ensure_output_dirs()


Mounted at /content/drive


In [ ]:
meta, arrays = RES.load_predictions(config.RESULTS_DIR / "predictions")
print(f"Loaded {len(meta)} prediction runs from disk.")

# Master list to collect consolidated statistical results
consolidated_results = []

ece_std_metric_fn = lambda a, b: MET.expected_calibration_error(a, b, config.ECE_N_BINS)
ece_adp_metric_fn = lambda a, b: MET.adaptive_calibration_error(a, b, config.ECE_N_BINS)


Loaded 48 prediction runs from disk.


## 2. Direction Contrast Tests (SQ1 and SQ2: Bootstrap Difference CIs)

In [ ]:
print("--- 1. Direction Contrasts (N2K vs K2N at Full Size 190) ---")

def evaluate_direction_contrast(metric_name, metric_fn):
    yt_k2n, yp_k2n = RES.pool_predictions(meta, arrays, direction="K2N", size=190)
    yt_n2k, yp_n2k = RES.pool_predictions(meta, arrays, direction="N2K")

    _, _, _, boot_k2n = ST.bootstrap_ci(yt_k2n, yp_k2n, metric_fn, n_boot=config.BOOTSTRAP_N, seed=config.SEED)
    _, _, _, boot_n2k = ST.bootstrap_ci(yt_n2k, yp_n2k, metric_fn, n_boot=config.BOOTSTRAP_N, seed=config.SEED)

    diff_val, ci_lo, ci_hi = ST.bootstrap_diff_ci(boot_n2k, boot_k2n, ci=config.BOOTSTRAP_CI)
    is_reliable = (ci_lo > 0 or ci_hi < 0)

    consolidated_results.append({
        "test": f"Direction Contrast ({metric_name})",
        "method": "Bootstrap Diff CI",
        "estimate": round(diff_val, 3),
        "ci_lo": round(ci_lo, 3),
        "ci_hi": round(ci_hi, 3),
        "p_value": np.nan,
        "significant": is_reliable
    })

    status_str = "reliable" if is_reliable else "not reliable"
    print(f"{metric_name:18s} | N2K - K2N Difference: {diff_val:+.3f}  95% CI [{ci_lo:+.3f}, {ci_hi:+.3f}] -> {status_str}")

evaluate_direction_contrast("AUROC", MET.auroc)
evaluate_direction_contrast("ECE (Equal-Width)", ece_std_metric_fn)
evaluate_direction_contrast("ECE (Adaptive)", ece_adp_metric_fn)


--- 1. Direction Contrasts (N2K vs K2N at Full Size 190) ---
AUROC              | N2K - K2N Difference: +0.025  95% CI [-0.055, +0.110] -> not reliable
ECE (Equal-Width)  | N2K - K2N Difference: -0.128  95% CI [-0.193, -0.067] -> reliable
ECE (Adaptive)     | N2K - K2N Difference: -0.112  95% CI [-0.180, -0.047] -> reliable


## 3. Architecture Contrast Within Direction (DeLong & McNemar Tests)

In [ ]:
print("\n--- 2. Architecture Contrasts (MobileNetV2 vs EfficientNet-B0 on N2K) ---")

def extract_architecture_predictions(direction, arch_name):
    """Extracts paired target vectors and ensembled probability predictions for an architecture."""
    selection = meta[(meta.direction == direction) & (meta.arch == arch_name)].sort_values(
        ["seed", "fold"] if "fold" in meta.columns else ["seed"]
    )
    prob_list = []
    y_reference = None

    for _, row in selection.iterrows():
        entry = arrays[row["stem"]]
        if y_reference is None:
            y_reference = entry["y_true"]
        prob_list.append(entry["y_prob"])

    return y_reference, np.mean(prob_list, axis=0)

y_true_n2k, prob_mobilenet = extract_architecture_predictions("N2K", "mobilenet_v2")
_, prob_efficientnet = extract_architecture_predictions("N2K", "efficientnet_b0")

# DeLong Test for AUROC Difference
auc_mob, auc_eff, p_val_delong = ST.delong_test(y_true_n2k, prob_mobilenet, prob_efficientnet)
print(f"DeLong Test (AUROC): MobileNetV2={auc_mob:.3f}, EfficientNet-B0={auc_eff:.3f} | p-value = {p_val_delong:.4f}")

consolidated_results.append({
    "test": "Architecture Contrast AUROC (N2K)",
    "method": "DeLong",
    "estimate": round(auc_mob - auc_eff, 3),
    "ci_lo": np.nan,
    "ci_hi": np.nan,
    "p_value": round(p_val_delong, 4),
    "significant": p_val_delong < 0.05
})

# McNemar Test for Classification Accuracy Difference (at threshold 0.5)
pred_mobilenet = (prob_mobilenet >= 0.5).astype(int)
pred_efficientnet = (prob_efficientnet >= 0.5).astype(int)

b_contingency, c_contingency, p_val_mcnemar = ST.mcnemar_test(y_true_n2k, pred_mobilenet, pred_efficientnet)
print(f"McNemar Test (Accuracy): Disagreements b={b_contingency}, c={c_contingency} | p-value = {p_val_mcnemar:.4f}")

consolidated_results.append({
    "test": "Architecture Contrast Accuracy (N2K)",
    "method": "McNemar",
    "estimate": np.nan,
    "ci_lo": np.nan,
    "ci_hi": np.nan,
    "p_value": round(p_val_mcnemar, 4),
    "significant": p_val_mcnemar < 0.05
})



--- 2. Architecture Contrasts (MobileNetV2 vs EfficientNet-B0 on N2K) ---
DeLong Test (AUROC): MobileNetV2=0.685, EfficientNet-B0=0.854 | p-value = 0.0000
McNemar Test (Accuracy): Disagreements b=60, c=125 | p-value = 0.0000


## 4. Size Interaction Test (Bootstrapped Direction-Gap Comparison)

In [ ]:
print("\n--- 3. Size Interaction Tests (Gap Size 190 - Gap Size 50) ---")

metrics_to_evaluate = [
    ("AUROC", MET.auroc),
    ("ECE (Equal-Width)", ece_std_metric_fn),
    ("ECE (Adaptive)", ece_adp_metric_fn)
]

for metric_name, metric_fn in metrics_to_evaluate:

    def compute_direction_gap_dist(size, m_fn=metric_fn):
        yt_k2n, yp_k2n = RES.pool_predictions(meta, arrays, direction="K2N", size=size)
        yt_n2k, yp_n2k = RES.pool_predictions(meta, arrays, direction="N2K")

        _, _, _, boot_k2n = ST.bootstrap_ci(yt_k2n, yp_k2n, m_fn, n_boot=config.BOOTSTRAP_N, seed=config.SEED)
        _, _, _, boot_n2k = ST.bootstrap_ci(yt_n2k, yp_n2k, m_fn, n_boot=config.BOOTSTRAP_N, seed=config.SEED)

        n_common = min(len(boot_k2n), len(boot_n2k))
        return boot_n2k[:n_common] - boot_k2n[:n_common]

    gap_50 = compute_direction_gap_dist(50)
    gap_190 = compute_direction_gap_dist(190)

    n_samples = min(len(gap_50), len(gap_190))
    interaction_dist = gap_190[:n_samples] - gap_50[:n_samples]

    ci_lo, ci_hi = np.percentile(interaction_dist, [2.5, 97.5])
    mean_interaction = float(interaction_dist.mean())
    is_reliable = (ci_lo > 0 or ci_hi < 0)

    consolidated_results.append({
        "test": f"Size x Direction Interaction ({metric_name})",
        "method": "Bootstrap Diff-of-Gaps CI",
        "estimate": round(mean_interaction, 3),
        "ci_lo": round(ci_lo, 3),
        "ci_hi": round(ci_hi, 3),
        "p_value": np.nan,
        "significant": is_reliable
    })

    status_str = "interaction present" if is_reliable else "no interaction"
    print(f"{metric_name:18s} Interaction Estimate: {mean_interaction:+.3f}  95% CI [{ci_lo:+.3f}, {ci_hi:+.3f}] -> {status_str}")



--- 3. Size Interaction Tests (Gap Size 190 - Gap Size 50) ---
AUROC              Interaction Estimate: -0.041  95% CI [-0.102, +0.020] -> no interaction
ECE (Equal-Width)  Interaction Estimate: -0.032  95% CI [-0.093, +0.034] -> no interaction
ECE (Adaptive)     Interaction Estimate: -0.018  95% CI [-0.082, +0.047] -> no interaction


## 5. Holm-Bonferroni Correction & Consolidated Table Export

In [ ]:
results_df = pd.DataFrame(consolidated_results)

# Apply Holm-Bonferroni step-down correction for tests with explicit p-values
p_value_mask = results_df["p_value"].notna()
if p_value_mask.any():
    p_vals = results_df.loc[p_value_mask, "p_value"]
    sorted_indices = p_vals.sort_values().index
    m_tests = len(p_vals)

    holm_p_adjusted = {}
    prev_adj_p = 0.0

    for rank, idx in enumerate(sorted_indices):
        raw_p = p_vals[idx]
        adj_p = min(1.0, (m_tests - rank) * raw_p)
        adj_p = max(adj_p, prev_adj_p)  # Enforce monotonicity
        holm_p_adjusted[idx] = round(adj_p, 4)
        prev_adj_p = adj_p

    results_df["p_holm"] = results_df.index.map(lambda i: holm_p_adjusted.get(i, np.nan))
else:
    results_df["p_holm"] = np.nan

# Organize columns for final report
column_order = ["test", "method", "estimate", "ci_lo", "ci_hi", "p_value", "p_holm", "significant"]
final_table = results_df[column_order]

print("\n=========================================================================")
print("                  CONSOLIDATED STATISTICAL TEST SUMMARY                  ")
print("=========================================================================")
print(final_table.to_string(index=False))

csv_output_path = config.RESULTS_DIR / "08_significance.csv"
final_table.to_csv(csv_output_path, index=False)
print(f"\nSaved consolidated test summary to {csv_output_path.name}")
print("\n[Pipeline Complete] Core empirical analysis (Notebooks 03-08) successfully finished.")



                  CONSOLIDATED STATISTICAL TEST SUMMARY                  
                                            test                    method  estimate  ci_lo  ci_hi  p_value  p_holm  significant
                      Direction Contrast (AUROC)         Bootstrap Diff CI     0.025 -0.055  0.110      NaN     NaN        False
          Direction Contrast (ECE (Equal-Width))         Bootstrap Diff CI    -0.128 -0.193 -0.067      NaN     NaN         True
             Direction Contrast (ECE (Adaptive))         Bootstrap Diff CI    -0.112 -0.180 -0.047      NaN     NaN         True
               Architecture Contrast AUROC (N2K)                    DeLong    -0.170    NaN    NaN      0.0     0.0         True
            Architecture Contrast Accuracy (N2K)                   McNemar       NaN    NaN    NaN      0.0     0.0         True
            Size x Direction Interaction (AUROC) Bootstrap Diff-of-Gaps CI    -0.041 -0.102  0.020      NaN     NaN        False
Size x Direction Inter

**Next:** notebook 09 (ood-musa).

Exploratory Out-of-Distribution response probing over-confidence on unseen Musa pathologies (TB / COVID-19).